# FoodTrip — Trợ lý AI: hiểu câu hỏi & yêu cầu lên lịch
Một mô hình PhoBERT, hai đầu ra: **ý định** của câu (lên lịch, sửa lịch, hỏi thanh toán, hoàn tiền, tour…) và **thông tin trong câu** (điểm đến, số ngày, ngày đi, ngân sách, số người, phương tiện, sở thích, món, quán, ngày trong lịch).
Lịch trình vẫn do thuật toán của FoodTrip xếp; mô hình chỉ đọc hiểu câu người dùng.
Chọn **Runtime → Change runtime type → T4 GPU** rồi chạy từng cell. Không cần API key hay đăng nhập Hugging Face.

## 1. Upload bộ chạy
Chọn **foodtrip-assistant-colab.zip** (tạo bằng `node scripts/package-assistant-colab.js`). File chỉ có code train và dữ liệu câu mẫu, không có .env hay dữ liệu tài khoản.

In [ ]:
from google.colab import files
from pathlib import Path
import io, json, zipfile

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("Chọn đúng một file foodtrip-assistant-colab.zip")
archive_name, archive_bytes = next(iter(uploaded.items()))
if not archive_name.lower().endswith(".zip"):
    raise ValueError("Cần file .zip")
WORK = Path("/content/foodtrip-assistant")
WORK.mkdir(exist_ok=True)
allowed = {"assistant_model.py", "train_assistant.py", "train_retriever.py", "test_assistant.py", "assistant_infer.py", "requirements-colab.txt", "ner_preprocessing.py",
           "data/train.jsonl", "data/validation.jsonl", "data/test.jsonl", "data/real_test.jsonl", "data/labels.json", "data/baseline.json",
           "data/faq_entries.jsonl", "data/faq_train.jsonl", "data/faq_validation.jsonl", "data/faq_test.jsonl", "data/faq_baseline.json"}
with zipfile.ZipFile(io.BytesIO(archive_bytes)) as archive:
    names = {p.filename for p in archive.infolist() if not p.is_dir()}
    if names != allowed or sum(p.file_size for p in archive.infolist()) > 20_000_000:
        raise ValueError(f"Nội dung ZIP không đúng bộ train; thiếu {sorted(allowed - names)}, thừa {sorted(names - allowed)}")
    for name in names:
        target = WORK / name
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(name))
del uploaded, archive_bytes
for split in ("train", "validation", "test", "real_test"):
    print(split, sum(1 for _ in (WORK / f"data/{split}.jsonl").open(encoding="utf-8")), "câu")
print("Mốc bộ luật:", json.loads((WORK / "data/baseline.json").read_text())["real_test"]["intent_accuracy"])
print("Mốc tìm câu trả lời bằng từ ngữ:", json.loads((WORK / "data/faq_baseline.json").read_text()))

## 2. Môi trường GPU riêng
Cài Python 3.12 và thư viện trong venv riêng (giống notebook NER). Chờ **GPU_SETUP_OK** rồi mới chạy tiếp.

In [ ]:
"""Paste this entire file into Colab's GPU environment cell (not a local run)."""
import os
import subprocess
import sys
from pathlib import Path


def run(command, *, cwd=None, env=None):
    """Stream both output channels so installation failures are visible in Colab."""
    process_env = dict(os.environ if env is None else env)
    process_env['PYTHONUNBUFFERED'] = '1'
    with subprocess.Popen(command, cwd=cwd, env=process_env,
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as process:
        for line in process.stdout:
            print(line, end='', flush=True)
        status = process.wait()
    if status:
        raise subprocess.CalledProcessError(status, command)


WORK = Path('/content/foodtrip-assistant')
if not (WORK / 'requirements-colab.txt').is_file():
    raise RuntimeError('Chạy cell upload foodtrip-assistant-colab.zip trước')
run(['nvidia-smi'])
TOOLS = Path('/content/foodtrip-uv-tools')
run([sys.executable, '-m', 'pip', 'install', '--disable-pip-version-check',
     '--target', str(TOOLS), '--upgrade', 'uv==0.9.5'])
uv_env = dict(os.environ)
uv_env['PYTHONPATH'] = str(TOOLS)
UV = [sys.executable, '-m', 'uv']
run(UV + ['python', 'install', '3.12'], env=uv_env)
# A different path preserves the incomplete environment from the failed cell.
ENV = Path('/content/foodtrip-assistant-py312')
PY = str(ENV / 'bin/python')
if not Path(PY).exists():
    run(UV + ['venv', '--python', '3.12', '--seed', str(ENV)], env=uv_env)
run([PY, '-c', 'import sys; print(sys.version); assert sys.version_info[:2] == (3, 12)'])
run([PY, '-m', 'pip', 'install', 'pip==25.0.1', 'setuptools==75.8.0', 'wheel', 'numpy==2.2.6'])
run([PY, '-m', 'pip', 'install', 'torch==2.6.0', '--index-url', 'https://download.pytorch.org/whl/cu124'])
run([PY, '-m', 'pip', 'install', '--no-build-isolation', '-r', str(WORK / 'requirements-colab.txt')])
run([PY, '-m', 'pip', 'check'])
run([PY, '-c', "import torch; print(torch.__version__); assert torch.cuda.is_available(), 'Chưa có GPU'; print(torch.cuda.get_device_name(0))"])
print('GPU_SETUP_OK — tiếp tục mục 3. Kiểm tra và smoke test', flush=True)

## 3. Kiểm tra và smoke test
Kiểm tra mã hóa/nhãn và dữ liệu, rồi chạy thử 3 bước train — **chưa phải train thật**. Có lỗi thì dừng.

In [ ]:
run([PY, str(WORK / "test_assistant.py"), "data"], cwd=WORK)
run([PY, str(WORK / "train_assistant.py"), "--data", "data", "--output", "runs/smoke", "--smoke"], cwd=WORK)
run([PY, str(WORK / "train_retriever.py"), "--data", "data", "--output", "runs/retriever-smoke", "--smoke"], cwd=WORK)

## 4. Train
Tối đa 10 epoch, dừng sớm theo validation (trung bình độ đúng ý định và F1 thông tin). Trên T4 thường khoảng 10–20 phút.

In [ ]:
if not (WORK / "runs/smoke/smoke-passed.json").exists():
    raise RuntimeError("Smoke test chưa qua")
run([PY, str(WORK / "train_assistant.py"), "--data", "data", "--output", "runs/assistant", "--export-onnx"], cwd=WORK)

## 5. Kết quả
So với bộ luật trên tập test (cách nói chưa gặp khi train) và 100 câu viết tay. Không chỉnh mô hình theo tập test.

In [ ]:
RESULT = WORK / "runs/assistant"
print((RESULT / "REPORT.md").read_text())
errors = json.loads((RESULT / "errors.json").read_text())
print(len(errors), "câu sai (ý định hoặc thông tin). 15 câu đầu:")
for e in errors[:15]:
    print("-", e["text"], "| đúng:", e["intent"], "| đoán:", e["predicted"])

## 5b. Train bộ tìm câu trả lời
Tinh chỉnh bộ mã hóa câu tiếng Việt để mỗi câu hỏi tìm đúng mục hỏi–đáp của FoodTrip. Vài phút trên T4. So với cách so khớp từ ngữ đang chạy và với bộ mã hóa chưa tinh chỉnh.

In [ ]:
if not (WORK / "runs/retriever-smoke/smoke-passed.json").exists():
    raise RuntimeError("Smoke test của bộ tìm câu trả lời chưa qua")
run([PY, str(WORK / "train_retriever.py"), "--data", "data", "--output", "runs/retriever", "--export-onnx"], cwd=WORK)
print((WORK / "runs/retriever/RETRIEVER.md").read_text())

## 6. Thử một câu

In [ ]:
run([PY, str(WORK / "assistant_infer.py"), str(RESULT / "onnx"), "cuối tuần này đi Đà Lạt 2 người, 3 triệu, thích cà phê"], cwd=WORK)

## 7. Tải mô hình về
Gói gồm hai mô hình ONNX (đọc câu, tìm câu trả lời) + tokenizer + cấu hình cho Hugging Face Space, và các báo cáo. Colab có thể xoá file khi hết phiên — tải ngay.

In [ ]:
bundle = Path("/content/foodtrip-assistant-trained.zip")
with zipfile.ZipFile(bundle, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for item in (RESULT / "onnx").rglob("*"):
        if item.is_file():
            archive.write(item, Path("onnx") / item.relative_to(RESULT / "onnx"))
    for name in ("metrics.json", "errors.json", "REPORT.md"):
        archive.write(RESULT / name, name)
    RETRIEVER = WORK / "runs/retriever"
    if (RETRIEVER / "onnx/retriever.onnx").exists():
        for item in (RETRIEVER / "onnx").rglob("*"):
            if item.is_file():
                archive.write(item, Path("retriever") / item.relative_to(RETRIEVER / "onnx"))
        for name in ("retriever-metrics.json", "retriever-errors.json", "RETRIEVER.md"):
            archive.write(RETRIEVER / name, name)
files.download(str(bundle))

## Sau khi train
Giải nén **foodtrip-assistant-trained.zip**, chép thư mục `onnx` vào `ml/assistant/space/model` rồi làm theo `ml/assistant/space/README.md` để đưa lên Hugging Face Space.
FoodTrip vẫn chạy bằng bộ luật cho tới khi Space được cấu hình (`ASSISTANT_MODEL_URL`).